# 82514 · Sesión S22 — Muestrear: PRM, RRT, OMPL y el brazo con una caja

**Bloque 7** · jueves 3 de diciembre de 2026 · 2 h  ·  IQS Universitat Ramon Llull

**Qué hace este cuaderno.** Muestrear en lugar de cuadricular. El PRM y el RRT, escritos en Python, en el almacén del curso, con la tasa de éxito en N semillas que pide la opción B de la entrega 3; RRT-Connect, el suavizado y el campo potencial con su mínimo local; los planificadores de OMPL, la biblioteca que usa MoveIt 2. Y el 2R del puesto de picking con una caja en la mesa: su espacio de configuraciones, las dos ramas de la inversa de S10 y lo que hará MoveIt 2 el jueves 10.

**Se apoya en:** Lynch y Park (2017); Corke (2023); Kavraki et al. (1996); LaValle (1998); Kuffner y LaValle (2000); Karaman y Frazzoli (2011); Khatib (1986); Sucan, Moll y Kavraki (2012); documentación de OMPL y de MoveIt 2.

**Cómo usarlo en clase.** Sigue el guion de la sesión S22 en las diapositivas del bloque 7. Ejecuta la celda de instalación una sola vez al empezar; en Colab tarda unos segundos. Las celdas marcadas **Ejercicio** son para que los trabajen los estudiantes: las soluciones están al final del cuaderno.

---

In [ ]:
# Ejecutar una sola vez al abrir el cuaderno. En Colab tarda unos segundos: instala OMPL 2.0.1.
import sys, time, math, importlib, importlib.metadata, subprocess
import numpy as np, scipy, matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

FIJAS = {'numpy': '2.4.6', 'scipy': '1.16.3', 'matplotlib': '3.10.0', 'ompl': '2.0.1'}   # las del bloque 7
EN_COLAB = 'google.colab' in sys.modules

def asegurar(pip_name, import_name):
    try:
        importlib.import_module(import_name)
    except ImportError:
        print('Instalando', pip_name, '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name], check=False)

asegurar('ompl==2.0.1', 'ompl')
try:
    from ompl import util as ou
    ou.setLogLevel(ou.LogLevel.LOG_WARN)
    ou.RNG.setSeed(2026)                  # una sola vez, y antes de crear nada
    from ompl import base as ob, geometric as og
    HAY_OMPL = True
except ImportError:                       # sin red, o sin OMPL para este Python: el resto del cuaderno funciona igual
    HAY_OMPL = False
    print('AVISO: no se ha podido instalar OMPL: las secciones 4 y 6 no se ejecutan; las demás, sí.')

plt.rcParams['figure.figsize'] = (11, 3.6)
IQS_AZUL, IQS_VERDE, GRIS, AMBAR, ROJO = '#1B2A80', '#1FA355', '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))

def coma(x, n=2):
    """Un número con coma decimal y signo menos, para imprimir."""
    return f'{x:.{n}f}'.replace('.', ',').replace('-', '−')

for nombre, version in (('numpy', np.__version__), ('scipy', scipy.__version__), ('matplotlib', matplotlib.__version__),
                        ('ompl', importlib.metadata.version('ompl') if HAY_OMPL else 'no instalada')):
    print(nombre, version, '' if version == FIJAS[nombre] else f'(probado con {FIJAS[nombre]})')
print('en Colab:', EN_COLAB)
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy 2.4; las cifras con semilla pueden cambiar un poco.')

> **Aviso para Colab.** Ejecuta la primera celda al abrirlo: instala OMPL 2.0.1 (`pip install ompl==2.0.1`, unos segundos) y usa numpy, scipy y matplotlib de Colab.
>
> **En clase:** nada; las diapositivas 59 a 67 enseñan sus cifras.
>
> **En casa:** todo el cuaderno, con los ejercicios 1 a 3; las soluciones están al final. Para la opción B de la entrega 3, que sale del cuaderno de S22 (el S22a y este) y del tutorial guiado de Nav2, las secciones 1 a 4 dan la tasa de éxito en N semillas y la comparación con OMPL.
>
> **No se sube:** este cuaderno no tiene tarea en Moodle.
>
> Los tiempos de cálculo dependen de la máquina; los de OMPL, además, del azar: cambian de una ejecución a otra.

## 0. El almacén y el espacio libre (diapositiva 51)

*Ejecuta esta celda sin detenerte.* El almacén del cuaderno S22a, con el robot inflado (r = 0,28 m) y las mismas búsquedas en rejilla, para comparar.

In [ ]:
# El almacén y las búsquedas en rejilla (como en el S22a).
import math, heapq
import numpy as np
from scipy.ndimage import distance_transform_edt, binary_erosion

RES, FILAS, COLS = 0.10, 90, 130             # celdas de 10 cm: 13 m en x, 9 m en y


def mapa_almacen():
    """La rejilla de ocupacion del almacen de B6 (True = ocupado). Fila = y/0,1; columna = x/0,1: la celda i esta
    centrada en i·0,1 m (es un cuadrado de 10 cm, de i·0,1 − 0,05 a i·0,1 + 0,05)."""
    occ = np.zeros((FILAS, COLS), bool)
    occ[0, :] = occ[-1, :] = True              # paredes exteriores
    occ[:, 0] = occ[:, -1] = True
    occ[42:47, :] = True                       # el muro central, de y = 4,2 a 4,7 m
    occ[42:47, 8:28] = False                   #   la puerta ancha, de x = 0,8 a 2,8 m
    occ[42:47, 92:100] = False                 #   la puerta estrecha, de x = 9,2 a 10,0 m
    occ[12:36, 40:47] = True                   # estanterias de la nave de abajo
    occ[12:36, 64:71] = True
    occ[58:78, 34:41] = True                   # estanterias de la nave de arriba
    occ[58:78, 58:65] = True
    occ[58:78, 82:89] = True
    return occ


def con_estrecha_cerrada(occ):
    """El mismo almacen con la puerta estrecha cerrada (un palé delante, como en S18)."""
    o = occ.copy()
    o[42:47, 92:100] = True
    return o


OCC = mapa_almacen()
DIST = distance_transform_edt(~OCC) * RES    # distancia de centro a centro de cada celda a la ocupada mas cercana, en m
EXT = (-RES / 2, COLS * RES - RES / 2, -RES / 2, FILAS * RES - RES / 2)   # para imshow: la celda i, centrada en i·0,1 m
MUELLE, PICKING, DERECHA = (0.6, 0.6), (8.3, 8.2), (11.5, 6.0)   # los puntos con nombre del almacen, en m
R_ROBOT, V_MAX, W_MAX = 0.28, 0.5, 1.0       # el robot del almacen: radio, 0,5 m/s y 1 rad/s


def celda(p):
    """De (x; y) en metros a (fila, columna) de la rejilla."""
    return (int(round(p[1] / RES)), int(round(p[0] / RES)))


def libre_para(occ, r):
    """El espacio libre para el CENTRO de un robot redondo de radio r: los obstaculos, inflados con r."""
    return distance_transform_edt(~occ) * RES > r


VECINOS = [(-1, 0, 1.0), (1, 0, 1.0), (0, -1, 1.0), (0, 1, 1.0),
           (-1, -1, math.sqrt(2)), (-1, 1, math.sqrt(2)), (1, -1, math.sqrt(2)), (1, 1, math.sqrt(2))]


def buscar(coste, ini, meta, peso=1.0, devolver_cerrada=False):
    """A* sobre la rejilla, con 8 vecinos. coste[v] es lo que cuesta entrar en v (inf = no se puede).
    peso = 0: Dijkstra (h = 0); 1: A* con la euclidea, admisible; 2: la heuristica doblada, que se pasa.
    Devuelve (camino, coste, nodos expandidos[, conjunto de expandidos])."""
    h = lambda v: peso * math.hypot(v[0] - meta[0], v[1] - meta[1])
    n, m = coste.shape
    g, padre = {ini: 0.0}, {}
    abierta, cerrada = [(h(ini), 0.0, ini)], set()   # OPEN, ordenada por f = g + h; CLOSED
    while abierta:
        _, gu, u = heapq.heappop(abierta)
        if u in cerrada:
            continue
        cerrada.add(u)
        if u == meta:
            break
        for di, dj, paso in VECINOS:
            v = (u[0] + di, u[1] + dj)
            if not (0 <= v[0] < n and 0 <= v[1] < m) or v in cerrada:
                continue
            c = coste[v]
            if not np.isfinite(c):
                continue
            ng = gu + paso * c
            if ng < g.get(v, np.inf):
                g[v] = ng
                padre[v] = u
                heapq.heappush(abierta, (ng + h(v), ng, v))
    if meta not in g:
        res = (None, np.inf, len(cerrada))
    else:
        cam, u = [meta], meta
        while u != ini:
            u = padre[u]
            cam.append(u)
        res = (cam[::-1], g[meta], len(cerrada))
    return res + (cerrada,) if devolver_cerrada else res


def frente_de_onda(coste, meta):
    """Dijkstra desde la meta hacia todas partes: el coste de cada celda hasta la meta (la transformada de distancia)."""
    D = np.full(coste.shape, np.inf)
    D[meta] = 0.0
    monton = [(0.0, meta)]
    while monton:
        d, u = heapq.heappop(monton)
        if d > D[u]:
            continue
        for di, dj, paso in VECINOS:
            v = (u[0] + di, u[1] + dj)
            if not (0 <= v[0] < coste.shape[0] and 0 <= v[1] < coste.shape[1]):
                continue
            if not np.isfinite(coste[v]):
                continue
            nd = d + paso * coste[v]
            if nd < D[v]:
                D[v] = nd
                heapq.heappush(monton, (nd, v))
    return D


def bajar(D, ini, max_pasos=20000):
    """El camino desde cualquier celda: bajar por el frente de onda, al vecino de menor D, hasta la meta (D = 0)."""
    cam, u = [ini], ini
    for _ in range(max_pasos):
        if D[u] == 0.0:
            return cam
        v = min(((u[0] + di, u[1] + dj) for di, dj, _ in VECINOS
                 if 0 <= u[0] + di < D.shape[0] and 0 <= u[1] + dj < D.shape[1]), key=lambda w: D[w])
        if D[v] >= D[u]:
            return None
        u = v
        cam.append(u)
    return None


def largo(cam):
    """Longitud de un camino de celdas, en metros."""
    p = np.array(cam, float) * RES
    return float(np.linalg.norm(np.diff(p, axis=0), axis=1).sum())


_CARAS = np.argwhere(OCC & ~binary_erosion(OCC, border_value=0)) * RES   # (y, x) de las celdas ocupadas del borde


def a_la_pared(xy, caras=None):
    """La distancia (m) de cada punto (x, y) a la cara del obstaculo mas cercano: cada celda ocupada es un cuadrado
    de 10 cm, y se mide hasta su borde, no hasta su centro (que esta 5 cm mas alla)."""
    p = np.atleast_2d(np.asarray(xy, float))
    caras = _CARAS if caras is None else caras
    out = np.empty(len(p))
    for k in range(0, len(p), 500):                              # por tandas, para no gastar memoria
        q = p[k:k + 500]
        dx = np.maximum(np.abs(q[:, None, 0] - caras[None, :, 1]) - RES / 2, 0.0)
        dy = np.maximum(np.abs(q[:, None, 1] - caras[None, :, 0]) - RES / 2, 0.0)
        out[k:k + 500] = np.hypot(dx, dy).min(axis=1)
    return out


def holgura(cam):
    """La distancia minima del centro del robot a la pared (a la cara del obstaculo) a lo largo del camino, en m."""
    c = np.array(cam, float)
    return float(a_la_pared(c[:, ::-1] * RES).min())


def puerta(cam):
    """Por que puerta cruza el muro un camino de celdas."""
    cols = [cc for f, cc in cam if 42 <= f <= 46]
    return "ancha" if cols and max(cols) < 40 else ("estrecha" if cols else "ninguna")


def dibuja_almacen(ax, libre=None, titulo=''):
    img = np.ones(OCC.shape + (3,))
    if libre is not None:
        img[~libre & ~OCC] = (0.96, 0.84, 0.82)
    img[OCC] = (0.29, 0.31, 0.39)
    ax.imshow(img, origin='lower', extent=EXT, interpolation='nearest')
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(titulo)


A, B = celda(MUELLE), celda(PICKING)
LIBRE = libre_para(OCC, R_ROBOT)
cam_rej, cst_rej, _ = buscar(np.where(LIBRE, 1.0, np.inf), A, B, 1)
print(f'la referencia, A* en la rejilla: {coma(cst_rej * RES)} m')

## 1. El PRM: construir una vez, consultar muchas (diapositiva 59)

**Construir:** N muestras libres al azar, y cada una unida a sus k = 8 vecinas más cercanas si el segmento que las une está libre (se comprueba cada 5 cm). **Consultar:** se enganchan el inicio y la meta al grafo y A* busca en él.

Es **probabilísticamente completo**: cuantas más muestras, más probable encontrar camino. La tasa de éxito se mide en N semillas: es lo que pide la opción B para un planificador de muestreo. Fíjate en las tandas de 20: con pocas semillas, la tasa baila; di siempre cuántas has usado.

In [ ]:
# El muestreo: PRM, RRT, RRT-Connect y el suavizado (el mismo código que las diapositivas).
def segmento_libre(libre, a, b, paso=0.5):
    """¿Esta libre el segmento a -> b (en celdas)? Se comprueba cada «paso» celdas: 5 cm."""
    n = max(2, int(math.hypot(b[0] - a[0], b[1] - a[1]) / paso) + 1)
    for t in np.linspace(0, 1, n):
        i, j = int(round(a[0] + t * (b[0] - a[0]))), int(round(a[1] + t * (b[1] - a[1])))
        if not (0 <= i < libre.shape[0] and 0 <= j < libre.shape[1]) or not libre[i, j]:
            return False
    return True


def prm_construir(libre, N, k=8, semilla=35):
    """El PRM, fase de construccion: N muestras libres al azar y cada una unida a sus k vecinas mas cercanas
    con un segmento libre. Es un grafo que no depende ni del inicio ni de la meta."""
    rng = np.random.default_rng(semilla)
    nodos = []
    while len(nodos) < N:
        i, j = int(rng.integers(0, libre.shape[0])), int(rng.integers(0, libre.shape[1]))
        if libre[i, j]:
            nodos.append((i, j))
    P = np.array(nodos, float)
    d2 = ((P[:, None, :] - P[None, :, :]) ** 2).sum(-1)
    aristas = {}
    for a in range(N):
        for b in np.argsort(d2[a])[1:k + 1]:
            b = int(b)
            if b in aristas.get(a, {}):
                continue
            if segmento_libre(libre, nodos[a], nodos[b]):
                w = float(math.sqrt(d2[a, b]))
                aristas.setdefault(a, {})[b] = w
                aristas.setdefault(b, {})[a] = w
    return nodos, aristas


def prm_consultar(libre, nodos, aristas, ini, meta, k=12):
    """El PRM, fase de consulta: enganchar inicio y meta a sus vecinos libres y buscar con A* en el grafo."""
    nodos = list(nodos) + [ini, meta]
    ar = {a: dict(v) for a, v in aristas.items()}
    i0, i1 = len(nodos) - 2, len(nodos) - 1
    P = np.array(nodos, float)
    for idx in (i0, i1):
        d = np.linalg.norm(P - P[idx], axis=1)
        for b in np.argsort(d)[1:k + 1]:
            b = int(b)
            if segmento_libre(libre, nodos[idx], nodos[b]):
                ar.setdefault(idx, {})[b] = float(d[b])
                ar.setdefault(b, {})[idx] = float(d[b])
    h = lambda v: float(np.linalg.norm(P[v] - P[i1]))
    g, padre, cerrada, monton = {i0: 0.0}, {}, set(), [(h(i0), 0.0, i0)]
    while monton:
        _, gu, u = heapq.heappop(monton)
        if u in cerrada:
            continue
        cerrada.add(u)
        if u == i1:
            break
        for v, w in ar.get(u, {}).items():
            if gu + w < g.get(v, np.inf):
                g[v] = gu + w
                padre[v] = u
                heapq.heappush(monton, (g[v] + h(v), g[v], v))
    if i1 not in g:
        return None
    ruta, u = [i1], i1
    while u != i0:
        u = padre[u]
        ruta.append(u)
    return [nodos[i] for i in ruta[::-1]]


def largo_poligonal(cam):
    """Longitud de una poligonal de puntos (fila, columna) en celdas, en metros."""
    c = np.array(cam, float)
    return float(np.linalg.norm(np.diff(c, axis=0), axis=1).sum() * RES)


def rrt(libre, ini, meta, eps=6.0, sesgo=0.06, radio_meta=6.0, max_iter=6000, semilla=35):
    """El RRT (Lynch y Park, 2017): q_muestra al azar (con un 6 % de las veces, la meta), q_cercano del arbol,
    avanzar eps celdas hacia la muestra hasta q_nuevo y anadirlo si el segmento esta libre.
    Devuelve (nodos, padres, camino o None, iteraciones)."""
    rng = np.random.default_rng(semilla)
    nodos, padres = [np.array(ini, float)], [-1]
    meta_ = np.array(meta, float)
    for it in range(max_iter):
        q_muestra = meta_ if rng.random() < sesgo else np.array([rng.uniform(0, libre.shape[0]), rng.uniform(0, libre.shape[1])])
        P = np.array(nodos)
        k = int(np.argmin(((P - q_muestra) ** 2).sum(1)))
        q_cercano = nodos[k]
        d = q_muestra - q_cercano
        n = np.linalg.norm(d)
        if n < 1e-9:
            continue
        q_nuevo = q_cercano + min(eps, n) * d / n
        if not segmento_libre(libre, tuple(q_cercano), tuple(q_nuevo)):
            continue
        nodos.append(q_nuevo)
        padres.append(k)
        if np.linalg.norm(q_nuevo - meta_) < radio_meta and segmento_libre(libre, tuple(q_nuevo), meta):
            nodos.append(meta_)
            padres.append(len(nodos) - 2)
            cam, u = [], len(nodos) - 1
            while u != -1:
                cam.append(nodos[u])
                u = padres[u]
            return nodos, padres, np.array(cam[::-1]), it + 1
    return nodos, padres, None, max_iter


def rrt_connect(libre, ini, meta, eps=6.0, max_iter=6000, semilla=35):
    """RRT-Connect (Kuffner y LaValle, 2000): dos arboles, uno desde el inicio y otro desde la meta. En cada
    iteracion uno crece hacia una muestra y el otro intenta llegar hasta el nodo nuevo; despues cambian los papeles.
    Devuelve (arbol_a, arbol_b, camino o None, iteraciones); cada arbol es (nodos, padres)."""
    rng = np.random.default_rng(semilla)
    A = ([np.array(ini, float)], [-1])
    B = ([np.array(meta, float)], [-1])

    def extender(T, q, hasta_llegar=False):
        nodos, padres = T
        while True:
            P = np.array(nodos)
            k = int(np.argmin(((P - q) ** 2).sum(1)))
            d = q - nodos[k]
            n = np.linalg.norm(d)
            if n < 1e-9:
                return k, True
            q_nuevo = nodos[k] + min(eps, n) * d / n
            if not segmento_libre(libre, tuple(nodos[k]), tuple(q_nuevo)):
                return None, False
            nodos.append(q_nuevo)
            padres.append(k)
            if n <= eps:
                return len(nodos) - 1, True
            if not hasta_llegar:
                return len(nodos) - 1, False

    def rama(T, k):
        nodos, padres = T
        out = []
        while k != -1:
            out.append(nodos[k])
            k = padres[k]
        return out

    for it in range(max_iter):
        q = np.array([rng.uniform(0, libre.shape[0]), rng.uniform(0, libre.shape[1])])
        ka, _ = extender(A, q)
        if ka is not None:
            kb, llego = extender(B, A[0][ka], hasta_llegar=True)
            if llego:
                ra, rb = rama(A, ka)[::-1], rama(B, kb)
                cam = np.array(ra + rb[1:])
                if np.allclose(cam[0], meta):
                    cam = cam[::-1]
                arboles = (A, B) if np.allclose(A[0][0], ini) else (B, A)
                return arboles[0], arboles[1], cam, it + 1
        A, B = B, A
    arboles = (A, B) if np.allclose(A[0][0], ini) else (B, A)
    return arboles[0], arboles[1], None, max_iter


def suavizar(libre, cam, intentos=200, semilla=0):
    """Suavizado por atajos: se eligen dos puntos del camino al azar y, si el segmento que los une esta libre,
    se quita lo de en medio."""
    rng = np.random.default_rng(semilla)
    c = [np.array(q, float) for q in cam]
    for _ in range(intentos):
        if len(c) < 3:
            break
        i, j = sorted(rng.choice(len(c), 2, replace=False))
        if j - i < 2:
            continue
        if segmento_libre(libre, tuple(c[i]), tuple(c[j])):
            c = c[:i + 1] + c[j:]
    return np.array(c)

In [ ]:
# La tasa de éxito del PRM en 100 semillas (de la 1000 a la 1099), con N muestras (unos treinta segundos).
for N in (30, 120, 250, 500):
    ok, L = [], []
    t0 = time.time()
    for sem in range(100):
        nodos, aristas = prm_construir(LIBRE, N, k=8, semilla=1000 + sem)
        cam = prm_consultar(LIBRE, nodos, aristas, A, B)
        ok.append(cam is not None)
        if cam is not None:
            L.append(largo_poligonal(cam))
    ok = np.array(ok)
    print(f'N = {N:3d}: {ok.sum():3d} de 100 con camino (en tandas de 20: {", ".join(str(ok[i:i + 20].sum()) for i in range(0, 100, 20))}); '
          f'{coma(np.mean(L))} m de media (de {coma(min(L))} a {coma(max(L))}); {coma((time.time() - t0) * 10, 0)} ms por semilla')

fig, axs = plt.subplots(1, 3, figsize=(12, 3.0))
for ax, N in zip(axs, (30, 120, 500)):
    nodos, aristas = prm_construir(LIBRE, N, k=8, semilla=102)
    cam = prm_consultar(LIBRE, nodos, aristas, A, B)
    dibuja_almacen(ax, LIBRE, f'N = {N}: ' + (f'{coma(largo_poligonal(cam))} m' if cam is not None else 'sin camino'))
    for a_, vs in aristas.items():
        for b_ in vs:
            if b_ > a_:
                ax.plot([nodos[a_][1] * RES, nodos[b_][1] * RES], [nodos[a_][0] * RES, nodos[b_][0] * RES], color='#9BA6DD', lw=0.6)
    if cam is not None:
        c = np.array(cam, float) * RES; ax.plot(c[:, 1], c[:, 0], color=AMBAR, lw=2)
plt.show()

## 2. El RRT: las muestras tiran del árbol (diapositiva 60)

Una consulta, un árbol desde el inicio. En cada vuelta: una **q_muestra** al azar (un 6 % de las veces, la meta), el nodo más cercano del árbol, **q_cercano**, y un paso de ε = 6 celdas (0,6 m) hacia la muestra hasta **q_nuevo**, que entra en el árbol si el tramo está libre:

q_nuevo = q_cercano + ε·(q_muestra − q_cercano)/‖q_muestra − q_cercano‖

No cuadricula el espacio: lo que cuesta es comprobar choques, no contar celdas. Por eso domina en alta dimensión. A cambio, ni es óptimo ni repetible.

In [ ]:
# Tres semillas, su suavizado y la tasa en 20 semillas.
for sem in (35, 7, 2026):
    nodos, padres, cam, its = rrt(LIBRE, A, B, semilla=sem)
    suave = suavizar(LIBRE, cam)
    print(f'semilla {sem:4d}: {coma(largo_poligonal(cam))} m, árbol de {len(nodos)} nodos, {its} vueltas; suavizado, {coma(largo_poligonal(suave))} m')
L, IT = [], []
for sem in range(20):
    _, _, cam, its = rrt(LIBRE, A, B, semilla=sem)
    if cam is not None:
        L.append(largo_poligonal(cam)); IT.append(its)
print(f'RRT, 20 semillas: {len(L)} de 20 con camino; mediana {coma(np.median(L))} m (de {coma(min(L))} a {coma(max(L))}); {int(np.median(IT))} vueltas de mediana')

nodos, padres, cam, its = rrt(LIBRE, A, B, semilla=35)
suave = suavizar(LIBRE, cam)
fig, ax = plt.subplots(figsize=(6, 4.2))
dibuja_almacen(ax, LIBRE, f'Semilla 35: {coma(largo_poligonal(cam))} m; suavizado, {coma(largo_poligonal(suave))} m')
for k, pk in enumerate(padres):
    if pk >= 0:
        ax.plot([nodos[k][1] * RES, nodos[pk][1] * RES], [nodos[k][0] * RES, nodos[pk][0] * RES], color='#9BA6DD', lw=1)
ax.plot(cam[:, 1] * RES, cam[:, 0] * RES, color=AMBAR, lw=2); ax.plot(suave[:, 1] * RES, suave[:, 0] * RES, '--', color=IQS_VERDE, lw=2)
plt.show()

## 3. RRT-Connect, y un atajo con trampa (diapositiva 61)

**RRT-Connect** (Kuffner y LaValle, 2000): dos árboles, uno desde cada punta, que crecen el uno hacia el otro. Es el planificador por defecto de MoveIt 2.

**El campo potencial** (Khatib, 1986): la meta atrae y las paredes repelen, y el robot baja por la suma con pasos de 5 cm, sin buscar nada. Es rapidísimo, pero puede pararse en un **mínimo local**.

In [ ]:
# RRT-Connect en 20 semillas.
L, IT = [], []
for sem in range(20):
    _, _, cam, its = rrt_connect(LIBRE, A, B, semilla=sem)
    if cam is not None:
        L.append(largo_poligonal(cam)); IT.append(its)
print(f'RRT-Connect, 20 semillas: {len(L)} de 20; mediana {coma(np.median(L))} m; {int(np.median(IT))} vueltas de mediana')

In [ ]:
# El campo potencial (el mismo código que las diapositivas).
def campo_potencial(dist, ini, meta, k_atr=1.0, k_rep=2.0, alcance=1.0, r=R_ROBOT, paso=0.05, n=3000):
    """El campo potencial (Khatib, 1986), en metros: la meta atrae con ½·k_atr·‖q − meta‖² y cada obstaculo
    repele con ½·k_rep·(1/ρ − 1/alcance)² si ρ = d − r < alcance. El robot baja por el gradiente con pasos de
    5 cm. Si en 100 pasos (5 m de marcha) no se ha movido ni 25 cm, esta atascado en un minimo local: se para.
    Devuelve la trayectoria (m) y si ha llegado (a menos de 10 cm de la meta)."""
    rho = np.maximum(dist - r, 0.02)
    Urep = np.where(rho < alcance, 0.5 * k_rep * (1 / rho - 1 / alcance) ** 2, 0.0)
    gy, gx = np.gradient(Urep, RES)                              # filas = y, columnas = x
    q, meta_, tray = np.array(ini, float), np.array(meta, float), [np.array(ini, float)]
    for _ in range(n):
        if np.linalg.norm(q - meta_) < 0.1:
            return np.array(tray), True
        if len(tray) > 100 and np.linalg.norm(tray[-1] - tray[-101]) < 0.25:
            return np.array(tray[:-100]), False                   # atascado: se queda donde empezo a dar vueltas
        f, c = int(round(q[1] / RES)), int(round(q[0] / RES))
        grad = k_atr * (q - meta_) + np.array([gx[f, c], gy[f, c]])
        g = np.linalg.norm(grad)
        if g < 1e-6:
            break
        q = q - paso * grad / g
        tray.append(q.copy())
    return np.array(tray), False


for ini in (MUELLE, (1.8, 6.0), (6.0, 6.0)):
    tray, llega = campo_potencial(DIST, ini, PICKING)
    fin = tray[-1]
    print(f'desde ({coma(ini[0], 1)}; {coma(ini[1], 1)}): ' + ('llega' if llega else f'se para en ({coma(fin[0])}; {coma(fin[1])})')
          + f' tras {coma(len(tray) * 0.05, 1)} m, a {coma(math.hypot(*(fin - np.array(PICKING))))} m del picking')

## 4. OMPL en el almacén (diapositivas 61 y 62)

OMPL es la biblioteca abierta de planificadores por muestreo que usa MoveIt 2. Se le da el espacio (aquí, (x; y) en metros), una función que dice si un estado es válido (aquí, si su celda está en 𝒞 libre) y el inicio y la meta, y se le pide un plan con un tiempo máximo. Dos detalles de su API: `RNG.setSeed` una sola vez y antes de crear nada (está en la primera celda), y **comprobar `haveExactSolutionPath()`**, porque `solve()` también da por buena una solución aproximada que no llega a la meta.

In [ ]:
# Cuatro planificadores de OMPL, con 1 s cada uno (unos segundos; RRT* usa el segundo entero).
def plan_almacen(planificador, tiempo=1.0, libre=LIBRE, ini=MUELLE, fin=PICKING):
    sp = ob.RealVectorStateSpace(2)
    lim = ob.RealVectorBounds(2); lim.setLow(0, 0.0); lim.setHigh(0, 12.9); lim.setLow(1, 0.0); lim.setHigh(1, 8.9)
    sp.setBounds(lim)
    ss = og.SimpleSetup(sp)
    ss.setStateValidityChecker(lambda s: bool(libre[int(round(s[1] / RES)), int(round(s[0] / RES))]))
    ss.getSpaceInformation().setStateValidityCheckingResolution(0.004)     # unos 6 cm: el 0,4 % de la diagonal del espacio (15,7 m)
    a_, b_ = sp.allocState(), sp.allocState()
    a_[0], a_[1] = ini; b_[0], b_[1] = fin
    ss.setStartAndGoalStates(a_, b_)
    ss.setPlanner(planificador(ss.getSpaceInformation()))
    t0 = time.time(); ss.solve(tiempo); dt = time.time() - t0
    if not ss.haveExactSolutionPath():
        return None, dt, None, None
    L0 = ss.getSolutionPath().length(); ss.simplifySolution()
    pa = ss.getSolutionPath()
    return L0, dt, pa.length(), np.array([(pa.getState(i)[0], pa.getState(i)[1]) for i in range(pa.getStateCount())])

RES_OMPL = {}
for nom, pl in ((('RRT', og.RRT), ('RRTConnect', og.RRTConnect), ('RRTstar', og.RRTstar), ('PRM', og.PRM)) if HAY_OMPL else ()):
    L0, dt, L1, pts = plan_almacen(pl)
    RES_OMPL[nom] = pts
    print(f'{nom:10s}: ' + (f'{coma(L0)} m -> {coma(L1)} m simplificado, en {coma(dt * 1000, 0)} ms' if L0 else f'sin solución exacta en {coma(dt, 2)} s'))
print(f'la rejilla con A*: {coma(cst_rej * RES)} m (solo gira de 45° en 45°)')
fig, ax = plt.subplots(figsize=(6, 4.2))
dibuja_almacen(ax, LIBRE, 'OMPL: RRT* (ámbar) y RRTConnect (azul), simplificados')
for nom, col in (('RRTstar', AMBAR), ('RRTConnect', IQS_AZUL)):
    if RES_OMPL.get(nom) is not None:
        ax.plot(RES_OMPL[nom][:, 0], RES_OMPL[nom][:, 1], color=col, lw=2)
plt.show()

## 5. El 2R con una caja: su espacio de configuraciones (diapositiva 52)

El 2R de B4 (a₁ = 0,35 m, a₂ = 0,20 m), con eslabones de 4 cm de ancho y límites de ±180° (no gira sin fin), en la mesa del puesto. La caja, de 10 × 10 cm y 0,20 m de alto, más alta que el brazo: x de 0,20 a 0,30 m e y de 0,10 a 0,20 m. La pieza de S17 en P = (0,4; 0,2) m. Se parte de la postura de espera, (90°; −60°).

Su espacio de configuraciones es el cuadrado (q₁; q₂). Se cuadricula a 2° (180 × 180 celdas) para dibujarlo y buscar con A*: con dos ejes todavía se puede (unos segundos).

In [ ]:
# El 2R, la caja y su espacio de configuraciones (el mismo código que las diapositivas).
A1, A2, R_ESL = 0.35, 0.20, 0.02             # el 2R de B4, con eslabones de 4 cm de ancho
CAJA = (0.20, 0.30, 0.10, 0.20)              # la caja: x de 0,20 a 0,30 m, y de 0,10 a 0,20 m (0,20 m de alto)
SIN_CAJA = (9.0, 9.0, 9.0, 9.0)              # una caja fuera del alcance: la escena sin la caja
P_PIEZA = np.array([0.4, 0.2])               # la pieza de S17, en el marco de la mesa
ESPERA = (90.0, -60.0)                       # la postura de espera, en grados


def inversa(p, codo):
    """La cinematica inversa del 2R (S10): codo = +1, codo abajo; −1, codo arriba. Grados."""
    x, y = p
    c2 = (x * x + y * y - A1 * A1 - A2 * A2) / (2 * A1 * A2)
    q2 = codo * math.acos(c2)
    q1 = math.atan2(y, x) - math.atan2(A2 * math.sin(q2), A1 + A2 * math.cos(q2))
    return math.degrees(q1), math.degrees(q2)


def eslabones(q1d, q2d):
    """Los puntos del 2R en la mesa: la base, el codo y la pinza (m)."""
    q1, q2 = math.radians(q1d), math.radians(q2d)
    codo = np.array([A1 * math.cos(q1), A1 * math.sin(q1)])
    return np.zeros(2), codo, codo + np.array([A2 * math.cos(q1 + q2), A2 * math.sin(q1 + q2)])


def _dist_seg_caja(a, b, caja, n=40):
    t = np.linspace(0, 1, n)[:, None]
    q = a + t * (b - a)
    dx = np.maximum(np.maximum(caja[0] - q[:, 0], 0), q[:, 0] - caja[1])
    dy = np.maximum(np.maximum(caja[2] - q[:, 1], 0), q[:, 1] - caja[3])
    return np.hypot(dx, dy).min()


def choca(q1d, q2d, caja=CAJA, radio=R_ESL):
    """¿Toca algun eslabon la caja? (en planta: la caja es mas alta que el brazo)."""
    b, c, p = eslabones(q1d, q2d)
    return _dist_seg_caja(b, c, caja) <= radio or _dist_seg_caja(c, p, caja) <= radio


QS = np.linspace(-180, 180, 180, endpoint=False) + 1.0   # la rejilla del espacio de configuraciones: celdas de 2°


def espacio_c(caja=CAJA, radio=R_ESL):
    """El C-obstaculo del 2R en la rejilla de 2°: obs[i, j] = choca(QS[i], QS[j])."""
    return np.array([[choca(a, b, caja, radio) for b in QS] for a in QS])


def indice(grados):
    return int(round((grados - 1.0 + 180) / 2.0)) % QS.size


def astar_articular(obs, q_ini, q_fin, dar_vuelta=False):
    """A* en el espacio articular (8 vecinos, celdas de 2°). Con dar_vuelta=False, los limites de ±180° cortan
    el cuadrado; con True, las articulaciones giran sin fin y el cuadrado es un toro.
    Devuelve (recorrido articular en grados o inf, nodos expandidos, camino de celdas)."""
    n = obs.shape[0]
    s, g = (indice(q_ini[0]), indice(q_ini[1])), (indice(q_fin[0]), indice(q_fin[1]))
    dif = lambda a, b: min(abs(a - b), n - abs(a - b)) if dar_vuelta else abs(a - b)
    h = lambda u: math.hypot(dif(u[0], g[0]), dif(u[1], g[1]))
    G, padre, abierta, cerrada = {s: 0.0}, {}, [(h(s), s)], set()
    while abierta:
        _, u = heapq.heappop(abierta)
        if u in cerrada:
            continue
        cerrada.add(u)
        if u == g:
            break
        for di in (-1, 0, 1):
            for dj in (-1, 0, 1):
                if di == dj == 0:
                    continue
                a, b = u[0] + di, u[1] + dj
                if dar_vuelta:
                    a, b = a % n, b % n
                elif not (0 <= a < n and 0 <= b < n):
                    continue
                v = (a, b)
                if obs[v] or v in cerrada:
                    continue
                ng = G[u] + math.hypot(di, dj)
                if ng < G.get(v, np.inf):
                    G[v], padre[v] = ng, u
                    heapq.heappush(abierta, (ng + h(v), v))
    if g not in G:
        return np.inf, len(cerrada), None
    cam, u = [g], g
    while u != s:
        u = padre[u]
        cam.append(u)
    return G[g] * 2.0, len(cerrada), cam[::-1]


def velocidad_articular(q2d, v=0.1, q1d=0.0):
    """La pinza del 2R en linea recta a v m/s, en direccion radial: |q̇| en °/s (S11: det J = a1·a2·sen q2)."""
    q1, q2 = math.radians(q1d), math.radians(q2d)
    J = np.array([[-A1 * math.sin(q1) - A2 * math.sin(q1 + q2), -A2 * math.sin(q1 + q2)],
                  [A1 * math.cos(q1) + A2 * math.cos(q1 + q2), A2 * math.cos(q1 + q2)]])
    p = np.array([A1 * math.cos(q1) + A2 * math.cos(q1 + q2), A1 * math.sin(q1) + A2 * math.sin(q1 + q2)])
    return math.degrees(np.linalg.norm(np.linalg.solve(J, v * p / np.linalg.norm(p))))

In [ ]:
# Las dos ramas de la inversa (S10), el C-obstáculo y A* en el cuadrado (unos segundos).
arriba, abajo = inversa(P_PIEZA, -1), inversa(P_PIEZA, +1)
print(f'codo arriba ({coma(arriba[0], 1)}°; {coma(arriba[1], 1)}°), libre: {not choca(*arriba)}; '
      f'codo abajo ({coma(abajo[0], 1)}°; {coma(abajo[1], 1)}°), libre: {not choca(*abajo)}')
t0 = time.time()
OBS = espacio_c()
print(f'C-obstáculo: {OBS.sum()} de {OBS.size} celdas, el {coma(100 * OBS.mean(), 1)} % (en {coma(time.time() - t0, 1)} s)')
for nom, meta in (('codo arriba', arriba), ('codo abajo', abajo)):
    L, nexp, _ = astar_articular(OBS, ESPERA, meta)
    Lv, _, _ = astar_articular(OBS, ESPERA, meta, dar_vuelta=True)
    print(f'espera -> {nom}: con ±180°, ' + (f'{coma(L, 1)}°' if np.isfinite(L) else 'sin camino') + f' ({nexp} nodos); girando sin fin, {coma(Lv, 1)}°')
L, _, _ = astar_articular(espacio_c(SIN_CAJA), ESPERA, abajo)
print(f'sin la caja, espera -> codo abajo: {coma(L, 1)}°')
fig, ax = plt.subplots(figsize=(4.6, 4.4))
ax.imshow(OBS.T, origin='lower', extent=(-180, 180, -180, 180), cmap='Greys', vmax=2)
for q, col, nom in ((ESPERA, IQS_AZUL, 'espera'), (arriba, IQS_VERDE, 'codo arriba'), (abajo, ROJO, 'codo abajo')):
    ax.plot(*q, 'o', color=col); ax.annotate(nom, q, (5, 5), textcoords='offset points', color=col)
ax.set_xlabel('q₁ (°)'); ax.set_ylabel('q₂ (°)'); ax.set_title('El cuadrado del 2R y su C-obstáculo')
plt.show()

## 6. Lo que hará MoveIt 2: el RRTConnect de OMPL sobre el 2R (diapositivas 66 y 67)

MoveIt 2 planifica con OMPL y, por defecto, con RRTConnect, comprobando cada estado contra la **escena de planificación**. Aquí, la escena es la caja: la función de validez es `choca`. Sin la caja en la escena, el plan sale **contra el aire**: se comprueba después frente a la caja de verdad.

Y la **línea recta**: llevar la pinza en recta a 0,1 m/s, en radial, cerca de q₂ = 0, donde el determinante del jacobiano de S11 se anula (det J = a₁·a₂·sen q₂).

Los tiempos de esta sección son de OMPL llamado desde Python en tu máquina; MoveIt 2, en C++, tarda menos (lo medís en S23). Lo que no cambia es si hay plan.

In [ ]:
# RRTConnect sobre el 2R, con 1 s, con la caja y sin ella.
def plan_2r(meta, caja=CAJA, tiempo=1.0):
    sp = ob.RealVectorStateSpace(2)
    lim = ob.RealVectorBounds(2); lim.setLow(-math.pi); lim.setHigh(math.pi); sp.setBounds(lim)
    ss = og.SimpleSetup(sp)
    ss.setStateValidityChecker(lambda s: bool(not choca(math.degrees(s[0]), math.degrees(s[1]), caja)))
    ss.getSpaceInformation().setStateValidityCheckingResolution(0.002)
    a_, b_ = sp.allocState(), sp.allocState()
    a_[0], a_[1] = map(math.radians, ESPERA); b_[0], b_[1] = map(math.radians, meta)
    ss.setStartAndGoalStates(a_, b_)
    ss.setPlanner(og.RRTConnect(ss.getSpaceInformation()))
    t0 = time.time(); ss.solve(tiempo); dt = time.time() - t0
    if not ss.haveExactSolutionPath():
        return None, dt, None
    ss.simplifySolution()
    pa = ss.getSolutionPath(); pa.interpolate(200)
    qs = [(math.degrees(pa.getState(i)[0]), math.degrees(pa.getState(i)[1])) for i in range(pa.getStateCount())]
    return math.degrees(pa.length()), dt, qs

for nom, meta, caja in ((('codo arriba, con la caja', arriba, CAJA), ('codo abajo, con la caja', abajo, CAJA),
                         ('codo abajo, sin la caja', abajo, SIN_CAJA)) if HAY_OMPL else ()):
    L, dt, qs = plan_2r(meta, caja)
    if L is None:
        print(f'{nom:26s}: sin plan exacto en {coma(dt, 2)} s')
    else:
        print(f'{nom:26s}: plan en {coma(dt * 1000, 0)} ms, {coma(L, 1)}°; frente a la caja de verdad, choca en '
              f'{sum(choca(*q) for q in qs)} de {len(qs)} puntos')
print()
for q2 in (60, 30, 10, 5, 2):
    print(f'q₂ = {q2:2d}°: det J = {coma(A1 * A2 * math.sin(math.radians(q2)), 4)} m²; |q̇| = {coma(velocidad_articular(q2), 0)} °/s para 0,1 m/s')

### Ejercicio 1 (en casa): cuántas vecinas en el PRM

Con N = 60 muestras, repite la tasa de éxito en 20 semillas con k = 4, 8 y 12 vecinas. ¿Qué cambia: el número de aristas, la tasa de éxito o la longitud?

In [ ]:
# Ejercicio 1: tu código aquí.

### Ejercicio 2 (en casa): el paso del RRT

Repite el RRT en 20 semillas con ε = 3, 6 y 12 celdas (0,3, 0,6 y 1,2 m). ¿Cómo cambian las vueltas y la longitud? ¿Por qué un paso grande descarta más tramos cerca de las puertas?

In [ ]:
# Ejercicio 2: tu código aquí.

### Ejercicio 3 (en casa): la caja, detrás de la pieza

Mueve la caja a x de 0,25 a 0,35 m e y de 0,30 a 0,40 m (`caja = (0.25, 0.35, 0.30, 0.40)`). ¿Siguen libres las dos ramas? ¿Hay camino a las dos? ¿Cuánto mide cada uno?

In [ ]:
# Ejercicio 3: tu código aquí.

## Soluciones

**Sección 1.** Con 30 muestras, 65 de 100 semillas encuentran camino; con 120, 86; con 250, 95; con 500, las 100. Con 500, el camino va de 12,36 a 14,49 m según la semilla (13,04 de media): en 20 de las 100 es más corto que el de la rejilla (12,66 m), que solo gira de 45° en 45°, pero nunca baja de los unos 11,9 m de RRT* (sección 4): no es óptimo. En tandas de 20, la tasa con 30 muestras baila de 11 a 15.

**Sección 2.** Semillas 35, 7 y 2026: 15,09, 17,00 y 15,83 m; suavizados, 13,15, 13,17 y 13,27 m. En 20 semillas, 20 de 20, con 16,69 m y 334 vueltas de mediana.

**Sección 3.** RRT-Connect: 20 de 20, 16,06 m y 134 vueltas de mediana. El campo potencial, desde el muelle, se para en (3,08; 3,13), a 7,28 m del picking.

**Sección 4.** RRT* llega a unos 11,9 m en 1 s, menos que los 12,66 m de la rejilla, que solo gira de 45° en 45°. RRT y RRTConnect dan caminos más largos que, simplificados, bajan mucho; sus cifras cambian en cada ejecución.

**Secciones 5 y 6.** El C-obstáculo es el 12,0 % del cuadrado. Al codo arriba, 42,6° con A*; al codo abajo, ningún camino con ±180° (girando sin fin, 327,5°; sin la caja, 170,5°). OMPL: al codo arriba, plan en unos milisegundos (40,6°); al codo abajo con la caja, ninguno en 1 s; sin la caja, la recta de 161,2°, que atraviesa la caja de verdad en 77 de 200 puntos. Para la recta a 0,1 m/s: 48, 93, 275, 549 y 1372 °/s con q₂ = 60°, 30°, 10°, 5° y 2°.

**Ejercicio 1.**

In [ ]:
# Solución del ejercicio 1
for k in (4, 8, 12):
    ok, L, ar = 0, [], []
    for sem in range(20):
        nodos, aristas = prm_construir(LIBRE, 60, k=k, semilla=1000 + sem)
        ar.append(sum(len(v) for v in aristas.values()) // 2)
        cam = prm_consultar(LIBRE, nodos, aristas, A, B)
        if cam is not None:
            ok += 1; L.append(largo_poligonal(cam))
    print(f'k = {k:2d}: {int(np.mean(ar))} aristas de media; {ok} de 20 con camino; ' + (f'{coma(np.mean(L))} m de media' if L else ''))

Más vecinas, más aristas (129, 224 y 294 de media), más éxito (9, 16 y 18 de 20) y caminos algo más cortos (16,1, 14,6 y 14,3 m): el grafo se queda menos en islas. Pero cada arista es una comprobación de choque más. Con 60 muestras, lo que manda es cuántas caen cerca de las puertas.

**Ejercicio 2.**

In [ ]:
# Solución del ejercicio 2
for eps in (3, 6, 12):
    L, IT = [], []
    for sem in range(20):
        _, _, cam, its = rrt(LIBRE, A, B, eps=eps, radio_meta=max(6.0, eps), semilla=sem)
        if cam is not None:
            L.append(largo_poligonal(cam)); IT.append(its)
    print(f'ε = {eps:2d} celdas: {len(L)} de 20; mediana {coma(np.median(L))} m; {int(np.median(IT))} vueltas de mediana')

Un paso pequeño necesita más vueltas: 565 con ε = 3 frente a 334 con 6 y 260 con 12; la longitud baja poco (16,77, 16,69 y 15,91 m de mediana), y los tres encuentran camino en las 20 semillas. Uno grande avanza deprisa por las naves, pero cerca de las puertas y de las estanterías descarta más tramos, porque el segmento entero tiene que estar libre.

**Ejercicio 3.**

In [ ]:
# Solución del ejercicio 3
caja = (0.25, 0.35, 0.30, 0.40)
obs = espacio_c(caja)
print(f'C-obstáculo: el {coma(100 * obs.mean(), 1)} %')
for nom, meta in (('codo arriba', arriba), ('codo abajo', abajo)):
    L, _, _ = astar_articular(obs, ESPERA, meta)
    print(f'{nom}: libre {not choca(*meta, caja)}; ' + (f'camino de {coma(L, 1)}°' if np.isfinite(L) else 'sin camino'))

Detrás de la pieza, la caja ocupa solo el 3,5 % del cuadrado y deja libres las dos ramas, con camino a las dos: 85,3° al codo arriba (rodea la caja) y 224,8° al codo abajo. Ahora no es la rama la que decide si hay camino, sino cuál es más corto: la escena cambia la respuesta.

---

## Para llevarse de esta sesión

- **Muestrear no cuadricula**: lo que cuesta es comprobar choques, no contar celdas. Por eso domina en alta dimensión.
- **El PRM es probabilísticamente completo**: la tasa de éxito en N semillas sube con las muestras (65 de 100 con 30; 100 de 100 con 500). Con 20 semillas, baila.
- **El RRT da un camino distinto en cada semilla**: ni óptimo ni repetible; suavizado, mucho mejor. RRT* recablea y, con tiempo, baja de la rejilla.
- **El campo potencial no es completo**: se para en un mínimo local.
- **En el brazo, la rama de la inversa decide si hay camino**, y sin la caja en la escena el plan sale contra el aire.

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*